# Filter Eval

Writes `articles_with_cat_filter_scored.csv` — the same file with one extra column `flag_score` (probability that the article should be flagged, i.e. P(flag=True)).

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn.functional import softmax
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm

MODEL_DIR   = Path("./models/flag-classifier/best")
INPUT_CSV   = Path("articles_with_attrib.csv")
OUTPUT_CSV  = Path("articles_with_attrib_filter_scored.csv")
OUTPUT_CSV_SAFE = Path("articles_with_attrib_filter_scored_safe.csv")
BATCH_SIZE  = 64
MAX_LENGTH  = 256

DEVICE = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")
print(f"Device: {DEVICE}")
print(f"Model : {MODEL_DIR.resolve()}")

Device: mps
Model : /Users/derenrich/projects/gacha/data_pipeline/models/flag-classifier/best


In [2]:
# Load model and tokenizer
if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Model not found at {MODEL_DIR.resolve()}. "
        "Run Filter Training.ipynb first."
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()
model.to(DEVICE)
print("Model loaded.")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Model loaded.


In [3]:
# Load articles CSV
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} rows, columns: {list(df.columns)}")

for col in ("name", "description"):
    if col not in df.columns:
        raise ValueError(f"Expected column '{col}' not found in {INPUT_CSV}")

texts = (
    "name: " + df["name"].fillna("").astype(str)
    + " description: " + df["description"].fillna("").astype(str)
).tolist()

Loaded 99,930 rows, columns: ['url', 'abstract', 'name', 'qid', 'description', 'banned_category', 'view_count', 'image_url', 'topic', 'percentile', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'image_license', 'image_credit', 'image_attribution_url']


In [4]:
# Run batched inference, collecting P(flag=True) for every row
scores: list[float] = []

with torch.inference_mode():
    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Scoring"):
        batch_texts = texts[start : start + BATCH_SIZE]
        enc = tokenizer(
            batch_texts,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        logits = model(**enc).logits          # (batch, 2)
        probs  = softmax(logits, dim=-1)      # (batch, 2)
        scores.extend(probs[:, 1].cpu().tolist())  # P(flag=True)

print(f"Scored {len(scores):,} rows.")

Scoring:   0%|          | 0/1562 [00:00<?, ?it/s]

Scored 99,930 rows.


In [5]:
# Attach score column and write output
df["flag_score"] = scores

df.to_csv(OUTPUT_CSV, index=False)
print(f"Written to {OUTPUT_CSV.resolve()}")

df[df.flag_score < 0.9].to_csv(OUTPUT_CSV_SAFE,index=False)

# Quick sanity check
print(df[["name", "description", "flag_score"]].sort_values("flag_score", ascending=False).head(10))

Written to /Users/derenrich/projects/gacha/data_pipeline/articles_with_attrib_filter_scored.csv
                               name  \
39774      Belgrade school shooting   
44627        Brighton hotel bombing   
5277                   Nikolas Cruz   
2965                  Elliot Rodger   
42947  Kyoto Animation arson attack   
64974        2016 Brussels bombings   
46144               Cinema Rex fire   
37550      2026 Minab school attack   
59646        2016 Nice truck attack   
56586       Mar Elias Church attack   

                                             description  flag_score  
39774                       2023 mass shooting in Serbia         1.0  
44627  1984 IRA assassination attempt on Margaret Tha...         1.0  
5277                  American mass murderer (born 1998)         1.0  
2965          British-American mass murderer (1991–2014)         1.0  
42947                   2019 mass murder in Kyoto, Japan         1.0  
64974          Islamic State suicide bombings in